# Práctica NLP: clasificación con BERT y un LLM pequeño

**Objetivo:** usar **dos modelos ya entrenados** sobre los mismos textos y comparar resultados.

1. **DistilBERT**: encoder que **ya tiene una cabeza entrenada** para sentimiento. Solo inferencia.
2. **Qwen 2.5 (0.5B)**: decoder que clasifica con *zero-shot* y *few-shot* prompting. **Sin entrenar pesos**.

Usamos **SST-2** (inglés; etiquetas `POSITIVE` y `NEGATIVE`).

[Dataset](https://huggingface.co/datasets/stanfordnlp/sst2) · [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [Qwen 0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)

In [ ]:
%pip -q install "transformers>=4.45,<5" "datasets>=3,<5" accelerate

## 1. Cargamos unos pocos textos

Seleccionamos observaciones de **validación**. Los ejemplos del prompt *few-shot* se extraerán solamente de **train**.

In [ ]:
import pandas as pd
import torch
from datasets import load_dataset

D = load_dataset("stanfordnlp/sst2")
N = 10  
val = D["validation"].shuffle(seed=42).select(range(N))
textos = list(val["sentence"])
reales = ["POSITIVE" if x == 1 else "NEGATIVE" for x in val["label"]]
display(pd.DataFrame({"texto": textos, "real": reales}))

## 2. DistilBERT: clasificación con un encoder

Cargamos un checkpoint que fue **preentrenado y luego ajustado para sentimiento**. No se entrena nada en esta práctica.

In [ ]:
from transformers import pipeline

bert = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1,
)
pred_bert = [r["label"] for r in bert(textos, truncation=True)]
display(pd.DataFrame({"texto": textos, "real": reales, "BERT": pred_bert}))

## 3. Qwen 0.5B: un LLM que genera una etiqueta

En vez de una cabeza de clasificación, vamos a **pedirle al LLM que escriba** `POSITIVE` o `NEGATIVE`.

En *zero-shot* damos instrucciones; en *few-shot*, agregamos ejemplos **dentro del prompt**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

modelo = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(modelo)
llm = AutoModelForCausalLM.from_pretrained(
    modelo,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to("cuda" if torch.cuda.is_available() else "cpu")
llm.eval()

def clasificar_llm(texto, ejemplos=()):
    instruccion = "Classify the sentiment of a movie review. Answer only POSITIVE or NEGATIVE."
    demos = (
             "".join(f"Review: {t}\nLabel: {y}\n\n" for t, y in ejemplos)
        ) if ejemplos else ""
    mensajes = [
        {"role": "system", "content": instruccion},
        {"role": "user", "content": demos + f"Review: {texto}\nLabel:"},
    ]
    x = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to(llm.device)
    with torch.inference_mode():
        y = llm.generate(x, max_new_tokens=12, do_sample=False, pad_token_id=tok.eos_token_id)
    respuesta = tok.decode(y[0, x.shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r"\b(POSITIVE|NEGATIVE)\b", respuesta.upper())
    return match.group(1) if match else "OTRO"

## 4. Zero-shot vs. few-shot

`K` indica **cuántos ejemplos por clase** agregamos al prompt (se toman de *train*). Cambiá `K` para experimentar.

In [ ]:
K = 1  # 0, 1 o 2 ejemplos por clase
train = D["train"].shuffle(seed=19)
ejemplos, cuenta = [], {0: 0, 1: 0}
for r in train:
    y = r["label"]
    if cuenta[y] < K:
        ejemplos.append((r["sentence"], "POSITIVE" if y == 1 else "NEGATIVE"))
        cuenta[y] += 1
    if min(cuenta.values()) >= K:
        break

pred_zero = [clasificar_llm(t) for t in textos]
pred_few = [clasificar_llm(t, ejemplos) for t in textos]
resultado = pd.DataFrame({"texto": textos, "real": reales,
                          "BERT": pred_bert, "zero-shot": pred_zero, "few-shot": pred_few})
display(resultado)
print("Aciertos en esta muestra pequeña (no son métricas robustas):")
for col in ["BERT", "zero-shot", "few-shot"]:
    print(f"{col}: {sum(resultado[col] == resultado['real'])}/{N}")

## 5. Experimentá

Cambiá el texto, `K` o la instrucción dentro de `clasificar_llm`. ¿En qué casos cambian las predicciones? ¿Few-shot siempre mejora?

In [ ]:
texto_nuevo = "The film was slow at first, but the ending was excellent."
print("BERT:", bert(texto_nuevo)[0]["label"])
print("LLM zero-shot:", clasificar_llm(texto_nuevo))
print("LLM few-shot:", clasificar_llm(texto_nuevo, ejemplos))